# Capítulo 5 — Classificação de Lesões Bucais com CNN

Este notebook implementa uma CNN para classificar radiografias odontológicas
em categorias de diagnóstico (saudável, cárie, lesão periapical).

In [ ]:
# ============================================================
# 1. Configuração inicial
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

from odontoia.data import clahe_equalization, resize_with_aspect, to_tensor
from odontoia.models.cnn import build_simple_cnn, predict_with_explanation
from odontoia.metrics.classification import accuracy, sensitivity, specificity

# Detecção de dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

In [ ]:
# ============================================================
# 2. Dataset sintético de lesões bucais
# ============================================================
from odontoia.tdd.test_helpers import create_test_radiograph, DentalTestData

# Gera dataset sintético com 3 classes
data = create_test_radiograph(size=(128, 128), n_classes=3, random_state=42)
train, test = data.split(test_size=0.3, random_state=42)

print(f'Train: {train.n_samples} amostras, Test: {test.n_samples} amostras')
print(f'Classes: {train.n_classes}')
print(f'Shape das imagens: {train.image_shape}')

In [ ]:
# ============================================================
# 3. Visualização das amostras
# ============================================================
class_names = ['Saudável', 'Cárie', 'Lesão Periapical']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    idx = i % len(train.images)
    ax.imshow(train.images[idx], cmap='gray')
    ax.set_title(f'{class_names[train.labels[idx]]}', fontsize=12)
    ax.axis('off')
plt.suptitle('Amostras do Dataset Odontológico', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 4. Construção do modelo
# ============================================================
model = build_simple_cnn(input_shape=(1, 128, 128), n_classes=3)
model = model.to(device)

# Otimizador e função de perda
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f'Parâmetros treináveis: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# ============================================================
# 5. Função de treinamento
# ============================================================
def train_epoch(model, images, labels, criterion, optimizer):
    model.train()
    optimizer.zero_grad()
    
    # Converte para tensor
    inputs = torch.stack([to_tensor(img, add_batch=True).squeeze(0) for img in images])
    targets = torch.tensor(labels, dtype=torch.long)
    
    inputs, targets = inputs.to(device), targets.to(device)
    
    outputs = model(inputs)
    loss = criterion(outputs, targets)
    
    loss.backward()
    optimizer.step()
    
    _, predicted = outputs.max(1)
    acc = predicted.eq(targets).sum().item() / targets.size(0)
    
    return loss.item(), acc

In [ ]:
# ============================================================
# 6. Ciclo de treinamento
# ============================================================
n_epochs = 10
history = {'loss': [], 'acc': []}

for epoch in range(n_epochs):
    loss, acc = train_epoch(model, train.images, train.labels, criterion, optimizer)
    history['loss'].append(loss)
    history['acc'].append(acc)
    print(f'Época {epoch+1:2d}/{n_epochs} | Loss: {loss:.4f} | Acurácia: {acc:.2%}')

In [ ]:
# ============================================================
# 7. Avaliação no teste
# ============================================================
model.eval()
with torch.no_grad():
    inputs = torch.stack([to_tensor(img, add_batch=True).squeeze(0) for img in test.images])
    targets = torch.tensor(test.labels, dtype=torch.long)
    inputs, targets = inputs.to(device), targets.to(device)
    outputs = model(inputs)
    _, predictions = outputs.max(1)

acc = (predictions == targets).sum().item() / targets.size(0)
print(f'Acurácia no teste: {acc:.2%}')

In [ ]:
# ============================================================
# 8. Predição com explicabilidade (Grad-CAM preview)
# ============================================================
# Testa com uma amostra
img_tensor = to_tensor(test.images[0], add_batch=True).to(device)
cls_idx, heatmap, conf = predict_with_explanation(model, img_tensor)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))
ax1.imshow(test.images[0], cmap='gray')
ax1.set_title(f'Original\nClasse: {class_names[test.labels[0]]}', fontsize=11)

ax2.imshow(heatmap, cmap='jet')
ax2.set_title(f'Grad-CAM', fontsize=11)

ax3.imshow(test.images[0], cmap='gray')
ax3.imshow(heatmap, cmap='jet', alpha=0.5)
ax3.set_title(f'Predito: {class_names[cls_idx]}\nConf: {conf:.1%}', fontsize=11)

for ax in [ax1, ax2, ax3]:
    ax.axis('off')
plt.tight_layout()
plt.show()

---
**Conclusão:** A CNN foi capaz de aprender a classificar lesões bucais.
No Capítulo 11 exploraremos mais o Grad-CAM para explicabilidade.